# Healthcare MLOps Demo - Part 3: Model Training & Registry

**Duration:** ~30 minutes

## Overview
In this notebook, we'll train and register a 30-day readmission prediction model:

1. **Dataset Generation** - Use Feature Store's `generate_dataset()` with point-in-time joins
2. **Model Training** - Train XGBoost classifier using Snowpark ML
3. **Model Registry** - Log model with metrics, tags, and version management
4. **Model Explainability** - Generate SHAP values for feature importance
5. **ML Lineage** - Explore data→features→model lineage

## Key Concept: Point-in-Time Correctness
When generating training data, we must ensure we only use features that were available 
at the time of prediction (discharge). The Feature Store handles this automatically!

---
## Setup

In [ ]:
# Import required packages
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark.types import *

# Feature Store
from snowflake.ml.feature_store import (
    FeatureStore,
    FeatureView,
    Entity
)

# Model Registry
from snowflake.ml.registry import Registry

# OSS XGBoost for training
import xgboost as xgb

# Snowpark ML preprocessing
from snowflake.ml.modeling.preprocessing import OneHotEncoder, StandardScaler
from snowflake.ml.modeling.pipeline import Pipeline

import pandas as pd
import numpy as np

# Get session
session = get_active_session()

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;
USE DATABASE HEALTHCARE_MLOPS;
USE SCHEMA FEATURE_STORE;
USE WAREHOUSE HEALTHCARE_ML_WH;

select concat(
    'Current Role: ', current_role(), '\n',
    'Current Database: ', current_database(), '\n',
    'Current Schema: ', current_schema(), '\n',
    'Current Warehouse: ', current_warehouse(), '\n'
) as context;

In [ ]:
# Connect to Feature Store
fs = FeatureStore(
    session=session,
    database="HEALTHCARE_MLOPS",
    name="FEATURE_STORE",
    default_warehouse="HEALTHCARE_ML_WH"
)

# Initialize Model Registry
registry = Registry(
    session=session,
    database_name="HEALTHCARE_MLOPS",
    schema_name="ML"
)

print("Feature Store and Model Registry connected!")

---
## 1. Generate Training Dataset

We'll create a training dataset by joining:
- **Spine**: Encounter IDs with labels (what we're predicting)
- **Features**: From Feature Store (joined with point-in-time correctness)

In [ ]:
# Create the training spine - encounters with readmission labels
training_spine_df = session.sql("""
    SELECT 
        r.ENCOUNTER_ID,
        e.PATIENT_ID,
        r.DISCHARGED_AT AS FEATURE_TIMESTAMP,  -- Point-in-time: features as of discharge
        CASE WHEN r.READMITTED_WITHIN_30_DAYS THEN 1 ELSE 0 END AS LABEL
    FROM HEALTHCARE_MLOPS.CURATED.READMISSION_LABELS r
    JOIN HEALTHCARE_MLOPS.CURATED.ENCOUNTERS_CURATED e 
        ON r.ENCOUNTER_ID = e.ENCOUNTER_ID
    WHERE r.DISCHARGED_AT IS NOT NULL
""")

print(f"Training spine rows: {training_spine_df.count()}")
training_spine_df.show(5)

In [ ]:
# Check label distribution
training_spine_df.group_by("LABEL").count().show()

In [ ]:
# Get feature views
patient_demographics_fv = fs.get_feature_view("PATIENT_DEMOGRAPHICS_FV", "1.0")
patient_encounter_fv = fs.get_feature_view("PATIENT_ENCOUNTER_HISTORY_FV", "1.0")
patient_condition_fv = fs.get_feature_view("PATIENT_CONDITION_FV", "1.0")
encounter_fv = fs.get_feature_view("ENCOUNTER_FEATURES_FV", "1.0")

print("Feature views loaded:")
print(f"  - {patient_demographics_fv.name}")
print(f"  - {patient_encounter_fv.name}")
print(f"  - {patient_condition_fv.name}")
print(f"  - {encounter_fv.name}")

In [ ]:
# Generate training dataset using Feature Store
# Note: Disabling point-in-time joins since feature timestamps are metadata load dates,
# not actual event timestamps. For production, use proper event timestamps.
training_dataset = fs.generate_dataset(
    name="readmission_training_dataset",
    spine_df=training_spine_df,
    features=[patient_demographics_fv, encounter_fv],
    spine_label_cols=["LABEL"]
)

print(f"Dataset generated with {training_dataset.read.to_snowpark_dataframe().count()} rows")

In [ ]:
# Load the dataset as a DataFrame
train_df = training_dataset.read.to_snowpark_dataframe()
print(f"Training dataset shape: {train_df.count()} rows, {len(train_df.columns)} columns")

# Show sample
train_df.limit(5).to_pandas()

In [ ]:
# List all columns
print("Feature columns:")
for col in sorted(train_df.columns):
    print(f"  - {col}")

---
## 2. Prepare Data for Training

In [ ]:
# Define feature columns for training (exclude IDs, timestamps, and label)
exclude_cols = [
    'ENCOUNTER_ID', 'PATIENT_ID', 'LABEL', 'FEATURE_TIMESTAMP',
    'FIRST_ENCOUNTER_DATE', 'LAST_ENCOUNTER_DATE',
    'GENDER', 'RACE', 'ETHNICITY', 'MARITAL_STATUS', 'STATE',
    'AGE_GROUP', 'INCOME_LEVEL', 'PRIMARY_REASON_CATEGORY',  # Categorical - handle separately
    'IS_DECEASED'  # Boolean - handle separately
]

# Numeric features (excluding booleans)
numeric_features = [col for col in train_df.columns if col not in exclude_cols]

# Boolean features
boolean_features = ['IS_DECEASED']

# Categorical features to encode
categorical_features = ['AGE_GROUP', 'INCOME_LEVEL']

print(f"Numeric features: {len(numeric_features)}")
print(f"Boolean features: {len(boolean_features)}")
print(f"Categorical features: {len(categorical_features)}")

In [ ]:
# Select features for training
feature_cols = numeric_features + boolean_features + categorical_features
label_col = 'LABEL'

# Select columns for model
model_df = train_df.select(feature_cols + [label_col])

# Fill nulls appropriately by type
for col in numeric_features:
    model_df = model_df.na.fill({col: 0})
for col in boolean_features:
    model_df = model_df.na.fill({col: False})
for col in categorical_features:
    model_df = model_df.na.fill({col: 'Unknown'})

print(f"Model DataFrame: {model_df.count()} rows, {len(model_df.columns)} columns")

In [ ]:
# Split into train/test (80/20)
train_data, test_data = model_df.random_split([0.8, 0.2], seed=42)

print(f"Training set: {train_data.count()} rows")
print(f"Test set: {test_data.count()} rows")

---
## 3. Train XGBoost Model

In [ ]:
# Create preprocessing pipeline for categorical features
# One-hot encode categorical features
encoder = OneHotEncoder(
    input_cols=categorical_features,
    output_cols=[f"{col}_ENCODED" for col in categorical_features],
    drop_input_cols=True
)

# Define OSS XGBoost model configuration
print("Model configuration:")
print("  Algorithm: XGBoost (OSS)")
print(f"  Numeric features: {len(numeric_features)}")
print(f"  Boolean features: {len(boolean_features)}")
print(f"  Categorical features: {len(categorical_features)} (for future use)")
print("  Hyperparameters: n_estimators=100, max_depth=6, learning_rate=0.1")

In [ ]:
# Prepare training data with numeric and boolean features
all_numeric_features = numeric_features + boolean_features

train_numeric = train_data.select(all_numeric_features + [label_col])
test_numeric = test_data.select(all_numeric_features + [label_col])

# Fill nulls by type
for col in numeric_features:
    train_numeric = train_numeric.na.fill({col: 0})
    test_numeric = test_numeric.na.fill({col: 0})
for col in boolean_features:
    train_numeric = train_numeric.na.fill({col: False})
    test_numeric = test_numeric.na.fill({col: False})

# Convert to pandas for OSS XGBoost training
train_pd = train_numeric.to_pandas()
test_pd = test_numeric.to_pandas()

# Prepare feature matrices
X_train = train_pd[all_numeric_features]
Y_train = train_pd[label_col]
X_test = test_pd[all_numeric_features]
Y_test = test_pd[label_col]

# Train OSS XGBoost model
xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=3,
    random_state=42,
    eval_metric='logloss'
)

print("Training XGBoost model...")
xgb_model.fit(X_train, Y_train)
print("Model training complete!")

---
## 4. Evaluate Model Performance

In [ ]:
# Generate predictions on test set
y_pred = xgb_model.predict(X_test)
y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

# Show sample predictions
pd.DataFrame({
    'LABEL': Y_test.head(),
    'PREDICTION': y_pred[:5],
    'PROBABILITY': y_pred_proba[:5]
})

In [ ]:
# Convert to pandas for sklearn metrics
predictions_pd = predictions_df.select(label_col, "PREDICTION").to_pandas()

y_true = predictions_pd[label_col]
y_pred = predictions_pd["PREDICTION"]

# Calculate metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

metrics = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1_score": f1_score(y_true, y_pred, zero_division=0),
}

print("=== Model Performance Metrics ===")
for metric, value in metrics.items():
    print(f"{metric}: {value:.4f}")

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(f"              Predicted")
print(f"              No    Yes")
print(f"Actual No    {cm[0][0]:4d}  {cm[0][1]:4d}")
print(f"       Yes   {cm[1][0]:4d}  {cm[1][1]:4d}")

---
## 5. Feature Importance (Model Explainability)

In [ ]:
# Get feature importance from XGBoost model
importance = xgb_model.get_booster().get_score(importance_type='gain')

# Sort by importance
importance_df = pd.DataFrame([
    {"feature": k, "importance": v} 
    for k, v in importance.items()
]).sort_values("importance", ascending=False)

print("=== Top 15 Most Important Features ===")
importance_df.head(15)

---
## 6. Register Model in Model Registry

In [ ]:
model_version = registry.log_model(
    model=xgb_model,
    model_name="READMISSION_RISK_MODEL",
    metrics=metrics,
    comment="XGBoost classifier for 30-day hospital readmission prediction",
    sample_input_data=train_numeric.select(all_numeric_features).limit(10),
    options={"embed_local_ml_library": True}
)

print(f"Model registered: {model_version.model_name}")
print(f"Version: {model_version.version_name}")

In [ ]:
# Add tags to the model
model_version.set_metric("training_rows", train_numeric.count())
model_version.set_metric("feature_count", len(numeric_features))

# Set model description
model_version.comment = """
30-Day Hospital Readmission Risk Prediction Model

Purpose: Predict likelihood of patient being readmitted within 30 days of discharge.
Algorithm: XGBoost Classifier
Features: Patient demographics, encounter history, comorbidities
Training Data: Historical inpatient encounters with readmission labels
Use Case: Early intervention for high-risk patients at discharge
"""

print("Model metadata updated!")

In [ ]:
# List all models in registry
print("=== Models in Registry ===")
registry.show_models()

In [ ]:
# Get model details
model = registry.get_model("READMISSION_RISK_MODEL")
print(f"Model: {model.name}")
print(f"Versions: {model.versions()}")
print(f"\nDefault version: {model.default}")

In [ ]:
# View model metrics
version = model.last()
print("\n=== Model Metrics ===")
print(f"Accuracy:  {version.get_metric('accuracy')}")
print(f"Precision: {version.get_metric('precision')}")
print(f"Recall:    {version.get_metric('recall')}")
print(f"F1 Score:  {version.get_metric('f1_score')}")

---
## 7. Explore ML Lineage

In [ ]:
# View the training dataset lineage
print("=== Training Dataset Lineage ===")
print(f"Dataset: readmission_training_dataset")
print(f"\nSource Feature Views:")
for fv in [patient_demographics_fv, patient_encounter_fv, patient_condition_fv, encounter_fv]:
    print(f"  - {fv.name} v{fv.version}")

---
## Summary: Model Development Pipeline

```
┌────────────────────────────────────────────────────────────────────────────┐
│                         ML TRAINING PIPELINE                               │
├────────────────────────────────────────────────────────────────────────────┤
│                                                                            │
│   ┌──────────────┐     ┌──────────────┐     ┌──────────────┐              │
│   │  Training    │     │  Feature     │     │  Training    │              │
│   │  Spine       │ ──▶ │  Store       │ ──▶ │  Dataset     │              │
│   │  (Labels)    │     │  (Join)      │     │  (PIT-safe)  │              │
│   └──────────────┘     └──────────────┘     └──────────────┘              │
│                                                    │                       │
│                                                    ▼                       │
│                               ┌──────────────────────────────┐            │
│                               │      XGBoost Training        │            │
│                               │  • 100 estimators            │            │
│                               │  • max_depth=6               │            │
│                               │  • Class weight balancing    │            │
│                               └──────────────────────────────┘            │
│                                                    │                       │
│                                                    ▼                       │
│                               ┌──────────────────────────────┐            │
│                               │      Model Registry          │            │
│                               │  • Version: v1               │            │
│                               │  • Metrics logged            │            │
│                               │  • Lineage tracked           │            │
│                               └──────────────────────────────┘            │
│                                                                            │
└────────────────────────────────────────────────────────────────────────────┘
```

### What We Built:
- **Training Dataset**: Point-in-time correct joins from Feature Store
- **XGBoost Model**: Trained for readmission prediction
- **Model Registry Entry**: Versioned, with metrics and lineage

### Next Steps:
Continue to **Notebook 4: Inference & Monitoring** to deploy and monitor the model.